# S4 — 時間序列 + EDA 實戰（解答版）

> **對應教材**：`S4_timeseries_eda.ipynb`
> **說明**：本檔為課堂練習的參考解答，含完整程式碼與重點教學提示。

---

## 使用說明

1. 請先執行 S3 產生 `../datasets/ecommerce/orders_enriched.csv`
2. 建議先自己嘗試作答，再對照本檔解答
3. 解答注重「能跑、能懂、符合 DA 日常寫法」，不追求最短一行


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    '../datasets/ecommerce/orders_enriched.csv',
    parse_dates=['order_date'],
)

# 沿用 S4 的時間衍生欄位
df['year']     = df['order_date'].dt.year
df['month']    = df['order_date'].dt.month
df['weekday']  = df['order_date'].dt.day_name()
df['year_mon'] = df['order_date'].dt.to_period('M')

print('資料形狀:', df.shape)
print('日期範圍:', df['order_date'].min(), '~', df['order_date'].max())
df.head(3)

> ### 💡 知識補給站 — EDA 系統化清單
> 
> EDA 不是隨便看看，有一份**系統化清單**可以確保不遺漏：
> 
> | 步驟 | 做什麼 | 工具 |
> |---|---|---|
> | 1. 形狀與型別 | 幾列幾欄？型別對嗎？ | `shape`, `dtypes`, `info()` |
> | 2. 缺值地圖 | 哪些欄位缺最多？ | `isna().sum()`, `isna().mean()` |
> | 3. 數值摘要 | min/max 合理嗎？有沒有離群值？ | `describe()` |
> | 4. 類別分布 | 有沒有拼寫錯誤或稀有類別？ | `value_counts()` |
> | 5. 相關性 | 哪些欄位高度相關？ | `corr()`（但記得：相關 ≠ 因果） |
> | 6. 時間趨勢 | 有沒有明顯的季節性或異常跳動？ | `resample().sum()` + 畫圖 |
> 
> 把這 6 步跑完，你就對資料有 **80% 的掌握**。
> 
> 延伸工具：`ydata-profiling`（原 pandas-profiling）— 一行指令就能產出完整 EDA 報告。
> 
> *延伸關鍵字：EDA checklist, exploratory data analysis, ydata-profiling, systematic analysis*

---
## 🟢 送分題解答

### Q1. 每個月份 (1~12) 的平均訂單金額

**思路**：用 `groupby('month')` 後對 `amount` 取 `mean`。這裡的 `month` 是 1~12 的整數（不分年），代表「同一個月份」的平均行為。

In [ ]:
avg_by_month = df.groupby('month')['amount'].mean().round(1)
print('📊 各月份平均訂單金額:')
print(avg_by_month)
print(f'\n🏆 平均金額最高月份: {avg_by_month.idxmax()} 月 (NT$ {avg_by_month.max():,.1f})')

### Q2. 訂單數最多的前 3 天

**思路**：先取 `order_date` 的日期部分（丟掉時間），用 `value_counts()` 取前 3。

In [ ]:
top3_days = df['order_date'].dt.date.value_counts().head(3)
print('📅 訂單數最多的前 3 天:')
print(top3_days)

---
## 🟡 核心題解答

### Q1. 每月訂單數的 3 個月移動平均

**思路**：
1. 先用 `groupby('year_mon')` 或 `resample('ME')` 算出每月訂單數
2. 再用 `.rolling(window=3).mean()` 取 3 個月移動平均

> **教學提示：`resample` vs `groupby(dt.to_period)`**
> - `resample('ME')`：需要 datetime index，會自動補上缺月（空月會變 0 或 NaN），適合做連續時序
> - `groupby(year_mon)`：不需要 index，結果是 PeriodIndex，**沒有資料的月份不會出現**
> - 做移動平均、趨勢圖用 `resample` 比較安全（避免缺月被跳過造成誤差）

In [ ]:
# 用 resample 版本（推薦：保證月份連續）
ts = df.set_index('order_date').sort_index()
monthly_orders = ts['order_id'].resample('ME').count()
monthly_orders_ma3 = monthly_orders.rolling(window=3).mean().round(1)

result = pd.DataFrame({
    '每月訂單數': monthly_orders,
    '3個月移動平均': monthly_orders_ma3,
})
print('📈 每月訂單數 + 3 個月移動平均:')
print(result)

### Q2. 2025 Q4 (10~12 月) 的總營收

**思路**：用 boolean mask 篩選日期區間 `2025-10-01` ~ `2025-12-31`，再對 `amount` 取 `sum`。

另一種寫法是用 `df['order_date'].dt.quarter == 4` + `year == 2025`，兩種都可。

> ### 💡 知識補給站 — 當心 Simpson's Paradox
> 
> 我們剛才看到「整體最強的品類 × 地區組合」，但如果**拆開不同時間段**看，結論可能完全反轉 — 這就是 **Simpson's Paradox**（辛普森悖論）。
> 
> **範例情境**：
> - 整體：A 品類營收 > B 品類 ✓
> - 但按季度拆開：**每一季 B 都贏 A** → 只是 A 在大季度有更多訂單，拉高了整體數字
> 
> **解法**：做分析時**多加一層 breakdown**（按時間、按地區、按客群），看看結論是否一致。
> 
> 口訣：「**整體趨勢 ≠ 分組趨勢，多切一刀才安心**」。
> 
> 這在醫學研究、A/B testing、商業分析中都是經典陷阱 — 聚合數據可以騙人。
> 
> *延伸關鍵字：Simpson's Paradox, ecological fallacy, stratified analysis, confounding variable*

In [ ]:
# 寫法 1：用日期區間篩選（最直觀）
mask = (df['order_date'] >= '2025-10-01') & (df['order_date'] <= '2025-12-31')
q4_revenue = df.loc[mask, 'amount'].sum()
print(f'💰 2025 Q4 總營收: NT$ {q4_revenue:,.0f}')

# 寫法 2：用 dt.quarter（更 Pythonic）
q4_revenue_v2 = df.loc[
    (df['year'] == 2025) & (df['order_date'].dt.quarter == 4),
    'amount'
].sum()
print(f'💰 2025 Q4 總營收 (驗證): NT$ {q4_revenue_v2:,.0f}')

### Q3. 每個品類的訂單金額中位數（排序）

**思路**：`groupby('category')['amount'].median()`，再 `sort_values(ascending=False)`。

> **為什麼用中位數而不是平均？** 中位數對極端值不敏感，更能反映「典型」客單價；平均數容易被少數大單拉高。

In [ ]:
median_by_cat = (
    df.groupby('category')['amount']
      .median()
      .sort_values(ascending=False)
      .round(1)
)
print('📊 各品類訂單金額中位數（由高到低）:')
print(median_by_cat)

> ### 💡 知識補給站 — Correlation ≠ Causation（相關不等於因果）
> 
> 剛才看到 `unit_price` 和 `amount` 的相關係數很高 — 但這**不代表**「提高單價就能增加營收」。
> 
> **經典反例**：冰淇淋銷量和溺水人數高度正相關 → 難道吃冰淇淋會溺水？不是！背後有**共同的隱藏變數 (confounding variable)**：夏天。天氣熱 → 買冰淇淋的人多 + 游泳的人多。
> 
> **在報告中**：
> - ✅ 安全的寫法：「A 與 B 呈正相關 (r = 0.85)」
> - ❌ 危險的寫法：「A **導致** B 增加」← 這需要**實驗**（如 A/B test）或進階統計方法才能證明
> 
> **因果推論**需要：控制實驗（A/B test）、工具變數（IV）、自然實驗、或 RCT — 這些是統計學和 ML 的進階主題。
> 
> *延伸關鍵字：correlation vs causation, confounding variable, A/B test, causal inference*

---
## 🔴 挑戰題解答 — 月度經營報告

**需求回顧**：每月一列，包含「月份 / 總訂單數 / 總營收 / 活躍顧客數 / 客單價 / MoM 成長率(%)」。

**解題步驟**：
1. `groupby('year_mon')` + `.agg({...})` 一次算出多個聚合
2. 重新命名欄位為中文（對上老闆的月報表）
3. 用 `總營收 / 總訂單數` 算客單價
4. 用 `.pct_change() * 100` 算 MoM 成長率
5. 依月份排序（PeriodIndex 本身就是可排序的）

> **教學提示：`pct_change()` 的陷阱**
> - 第一列一定是 `NaN`（沒有上一個月可以比）
> - 要確保資料**已按時間排序**，否則算出來的成長率是亂的
> - 單位是「倍數」不是「百分比」，要顯示成百分比要 `* 100`

In [ ]:
# Step 1: 一次聚合多個欄位
monthly_report = (
    df.groupby('year_mon')
      .agg(
          總訂單數=('order_id', 'count'),
          總營收=('amount', 'sum'),
          活躍顧客數=('customer_id', 'nunique'),
      )
      .sort_index()  # 先確保按月份排序，pct_change 才正確
)

# Step 2: 算客單價
monthly_report['客單價'] = (
    monthly_report['總營收'] / monthly_report['總訂單數']
).round(1)

# Step 3: 算 MoM 成長率 (%)
monthly_report['MoM成長率(%)'] = (
    monthly_report['總營收'].pct_change() * 100
).round(2)

# Step 4: 把 index 變成一個「月份」欄位，輸出更像報表
monthly_report = monthly_report.reset_index().rename(columns={'year_mon': '月份'})
monthly_report['月份'] = monthly_report['月份'].astype(str)

print('📋 月度經營報告:')
monthly_report

In [ ]:
# 延伸：找出成長率最高/最低的月份（忽略第一個月的 NaN）
valid = monthly_report.dropna(subset=['MoM成長率(%)'])
best_mom  = valid.loc[valid['MoM成長率(%)'].idxmax()]
worst_mom = valid.loc[valid['MoM成長率(%)'].idxmin()]

print(f"🚀 成長最猛: {best_mom['月份']}  (+{best_mom['MoM成長率(%)']}%)")
print(f"📉 衰退最多: {worst_mom['月份']}  ({worst_mom['MoM成長率(%)']}%)")

---
## 解題回顧

| 題目 | 關鍵 API | 常見錯誤 |
|---|---|---|
| 送分 Q1 | `groupby('month').mean()` | 把 `year_mon` 和 `month` 搞混 |
| 送分 Q2 | `dt.date.value_counts()` | 忘了 `.dt.date`，結果每筆時間戳都是 unique |
| 核心 Q1 | `resample('ME') + rolling(3)` | 用 `groupby` 跳過缺月導致移動平均失準 |
| 核心 Q2 | boolean mask + `sum()` | 日期字串寫錯格式 |
| 核心 Q3 | `groupby.median().sort_values()` | 用 mean 被極端值干擾 |
| 挑戰題 | `agg + pct_change` | 忘了先 `sort_index` 導致 MoM 錯誤 |

**下一步**：S5 會把這份月報表畫成長條圖 + 折線圖，變成可以直接塞進簡報的視覺化成果。
